In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
pd.options.display.max_columns = None
import sklearn
import scipy
import scipy.stats as stats
from scipy.stats import skew,boxcox_normmax, zscore
from scipy.special import boxcox1p
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer,KNNImputer
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, make_scorer, mean_absolute_error 
from sklearn.model_selection import KFold, RandomizedSearchCV
from mlxtend.regressor import StackingCVRegressor
from multiprocessing import cpu_count
# from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns
from catboost import CatBoostRegressor

In [2]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold


In [128]:
df_train = pd.read_csv('../data/train.csv', index_col ='Id')
df_test = pd.read_csv('../data/test.csv', index_col ='Id')

In [129]:
target = 'SalePrice'

In [130]:
data = pd.concat([df_train,df_test])

In [131]:
## Combine some of the variables together to add value and decrease number of features
data['TotalBath'] = data[['FullBath', 'BsmtFullBath', 'HalfBath', 'BsmtHalfBath']].fillna(0).dot([1, 1, 0.5, 0.5])
data['TotalSF'] = data[['TotalBsmtSF' , '1stFlrSF' , '2ndFlrSF']].fillna(0).dot([1, 1, 1])
data['TotalPorch'] = data[['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']].fillna(0).dot([1, 1, 1, 1])

data = data.drop(columns = ['FullBath', 'BsmtFullBath', 'HalfBath', 'BsmtHalfBath', 'TotalBsmtSF' , '1stFlrSF' , '2ndFlrSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch'])

In [132]:
data['PoolQC'] = np.where(data['PoolQC'].isna(), 0, 1)
data['Fence'] = np.where(data['Fence'].isna(), 0, 1)
data['MiscFeature'] = np.where(data['MiscFeature'].isna(), 0, 1)
data['Alley'] = np.where(data['Alley'].isna(), 0, 1)

data['LuxuryFeature'] =  data[['PoolQC', 'Fence', 'MiscFeature', 'Fireplaces', 'Alley']].dot([1, 1, 1, 1, 1])

data = data.drop(columns = ['PoolQC', 'Fence', 'MiscFeature', 'MiscVal', 'Fireplaces', 'FireplaceQu', 'Alley'])

In [133]:
numerical_features = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = data.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

In [134]:
numerical_features.remove(target)

In [135]:
data[categorical_features].isna().mean()[data[categorical_features].isna().mean()>0]

MSZoning        0.001370
Utilities       0.000685
Exterior1st     0.000343
Exterior2nd     0.000343
MasVnrType      0.605002
BsmtQual        0.027749
BsmtCond        0.028092
BsmtExposure    0.028092
BsmtFinType1    0.027064
BsmtFinType2    0.027407
Electrical      0.000343
KitchenQual     0.000343
Functional      0.000685
GarageType      0.053786
GarageFinish    0.054471
GarageQual      0.054471
GarageCond      0.054471
SaleType        0.000343
dtype: float64

In [136]:
data[numerical_features].isna().mean()[data[numerical_features].isna().mean()>0]

LotFrontage    0.166495
MasVnrArea     0.007879
BsmtFinSF1     0.000343
BsmtFinSF2     0.000343
BsmtUnfSF      0.000343
GarageYrBlt    0.054471
GarageCars     0.000343
GarageArea     0.000343
dtype: float64

In [137]:
features_impute_with_median = ['LotFrontage', 'MasVnrArea']
features_impute_with_mode = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd','Electrical', 'KitchenQual', 'Functional','SaleType']
features_impute_with_zero = ['GarageYrBlt', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'GarageCars', 'GarageArea']
feaures_impute_with_none = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish', 
                            'GarageQual', 'GarageCond']

In [138]:
data[features_impute_with_zero].fillna(0, inplace=True)
data[feaures_impute_with_none].fillna('None', inplace=True)

/var/folders/40/vgskp5ms6k56tfv_n3ct_kkw0000gn/T/ipykernel_2259/633284011.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[features_impute_with_zero].fillna(0, inplace=True)
/var/folders/40/vgskp5ms6k56tfv_n3ct_kkw0000gn/T/ipykernel_2259/633284011.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[feaures_impute_with_none].fillna('None', inplace=True)


In [139]:
for col in features_impute_with_median:
    data[col] = data.groupby('Neighborhood')[col] \
               .transform(lambda grp: grp.fillna(grp.median()))

for col in features_impute_with_mode:
    data[col] = data.groupby('Neighborhood')[col] \
               .transform(lambda grp: grp.fillna(
                    grp.mode().iat[0] if not grp.mode().empty 
                    else data[col].mode().iat[0]
               ))

In [140]:
qual_features = [x for x in categorical_features if 'Qual' in x]
cond_features = [x for x in categorical_features if 'Cond' in x]

In [141]:
[x for x in categorical_features if 'QC' in x]

['HeatingQC']

In [ ]:
'''GLQ	Good Living Quarters
       ALQ	Average Living Quarters
       BLQ	Below Average Living Quarters	
       Rec	Average Rec Room
       LwQ	Low Quality
       Unf	Unfinshed
       NA	No Basement'''

In [ ]:
'''Ex	Excellent
       Gd	Good
       TA	Typical - slight dampness allowed
       Fa	Fair - dampness or some cracking or settling
       Po	Poor - Severe cracking, settling, or wetness
       NA	No Basement'''